# Segmentation code using StarDist and StarDist-OPP
This notebook is based on the example code provided by the [StarDist project](https://github.com/stardist/stardist), with modifications to incorporate [StarDist-OPP](https://github.com/gatoniel/merge-stardist-masks)([Jelli et al., 2023](https://doi.org/10.1111/mmi.15064)) for improved cell segmentation.

The workflow includes image preparation, cell segmentation, and visualization of segmentation results.

The original StarDist code is distributed under the [BSD 3-Clause License](https://github.com/stardist/stardist/blob/main/LICENSE.txt).

## 0. Environment Setup

This notebook was developed using Python 3.9.19 on macOS (Apple Silicon: M3/M4) with [Miniforge](https://github.com/conda-forge/miniforge?utm_source=chatgpt.com).

Create a conda environment and install the required packages:

```bash
conda create -n stardist-env python=3.9.19
conda activate stardist-env

pip install tensorflow==2.14 tensorflow-metal==1.1.0 stardist==0.9.1
conda install ipykernel natsort=8.4.0 libcxx=20.1.3
pip install merge-stardist-masks
```

## 1. Import libraries and load the trained model

In [ ]:
# Standard libraries
import os
import sys
from glob import glob

# Numerical computation and visualization
import numpy as np
import tensorflow as tf
import matplotlib
import matplotlib.pyplot as plt

# Image processing
from PIL import Image, ImageOps
from tifffile import imread
from skimage.io import imsave
from skimage.measure import regionprops
from skimage.feature import canny
from skimage.color import gray2rgb
from skimage.exposure import rescale_intensity

# Utilities
from natsort import natsorted
from csbdeep.utils import Path, normalize
from csbdeep.io import save_tiff_imagej_compatible

# StarDist
from stardist import random_label_cmap, _draw_polygons, export_imagej_rois
from stardist.models import StarDist2D
from merge_stardist_masks.naive_fusion import naive_fusion

# Visualization settings
matplotlib.rcParams["image.interpolation"] = "none"
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Random seed and label colormap
np.random.seed(6)
lbl_cmap = random_label_cmap()

# Check GPU availability
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices("GPU")))

In [ ]:
# Load the model. Set "demo_model = True" to use the demo model.
demo_model = False
if demo_model:
    print (
        "NOTE: This is loading a previously trained demo model!\n"
        "      Please set the variable 'demo_model = False' to load your own trained model.",
        file=sys.stderr, flush=True
    )
    model = StarDist2D.from_pretrained('2D_demo')
else:
    model = StarDist2D(None, name='241128_PY1', basedir='models')
None;

## 2. Prepare Input Images

In [ ]:
def complement_and_save_images(input_folder, output_folder):
    # Get all TIFF images in the input folder
    image_files = [filename for filename in os.listdir(input_folder) if filename.endswith(".tiff") or filename.endswith(".tif")]
    image_files = natsorted(image_files)
    num_images = len(image_files)
    print(num_images)

    # Process all images in the folder
    for index, filename in enumerate(image_files):
        # Load the image
        img = Image.open(os.path.join(input_folder, filename))
        width, height = img.size
        
       # Calculate the number of missing pixels
        missing_height = max(0, 1040 - height)
        missing_width = max(0, 1536 - width)

        # Pad missing regions with zero-intensity pixels
        if missing_height > 0 or missing_width > 0:
            padding = (0, 0, missing_width, missing_height)
            img = ImageOps.expand(img, padding, fill=0)
        
        width, height = img.size
            
        # Check whether the image has the expected dimensions
        if width != 1536 or height != 1040:
            print(f"Warning: {filename} does not have the expected size (1536x1040). Skipping...")
            continue

        # Generate a sequential filename starting from 001
        output_filename = f"{index + 1:03d}.tif"
        output_path = os.path.join(output_folder, output_filename)
        img.save(output_path)

In [ ]:
input_folder = "./1_images"
output_folder = "./2_images_complement"
os.makedirs(output_folder, exist_ok=True)

complement_and_save_images(input_folder, output_folder)

## 3. Perform Cell Segmentation

In [ ]:
# Perform prediction and save the results for all images in the directory.

# Set the paths for the input images and output directories.
image_dir = './2_images_complement' # Directory containing the input images
label_dir = './3_label_img' # Directory for saving labeled images
os.makedirs(label_dir, exist_ok=True)

# Index of the image to display (starting from 1)
display_image_index = 1

# Get a list of all image files in the directory
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.tif', '.tiff'))]

# Perform prediction for all images in the directory
for i, filename in enumerate(image_files, start=1):
    filepath = os.path.join(image_dir, filename)
    X = imread(filepath)
    img = normalize(X, 1, 99.8)

    labels, (probs, dists) = model.predict_instances(img, return_predict=True) # Predict instances and obtain probability and distance maps
    grid = model.config.grid # Get grid information
    labels_fused = naive_fusion(dists, probs, grid=grid, prob_thresh=0.3) # Merge masks using the naive_fusion function(StarDist-OPP)
     
    # Save the labeled image
    label_path = os.path.join(label_dir, os.path.splitext(filename)[0] + '_labels.tif')
    save_tiff_imagej_compatible(label_path, labels_fused, axes='YX')
       
    # Display the result only for the specified image
    if i == display_image_index:
        plt.figure(figsize=(8, 8))
        plt.imshow(img, clim=(0, 1), cmap='gray')
        plt.imshow(labels_fused, cmap='viridis', alpha=0.5)
        plt.axis('off')
        plt.show()

## 4. Visualize Segmentation Results

In [ ]:
# Set the paths for the input images and output directories.
image_dir = './2_images_complement' # Directory containing the original images
label_dir = './3_label_img' # Directory containing the labeled images
output_edge_dir = './4_label_img_edges' # Directory for saving images with overlaid edges
os.makedirs(output_edge_dir, exist_ok=True)

# Get a list of all labeled image files in the directory
label_files = [f for f in os.listdir(label_dir) if f.endswith(('.tif', '.tiff'))]

# Overlay edges on each original image and save the result
for filename in label_files:
    # Load the labeled image
    label_path = os.path.join(label_dir, filename)
    labels = imread(label_path)
    
    # Load the original image
    original_path = os.path.join(image_dir, os.path.splitext(filename)[0].replace('_labels', '') + '.tif')
    original_image = imread(original_path)
    
    # Detect edges
    edges = canny(labels > 0)
    
    # Overlay edges on the original image
    original_image_normalized = rescale_intensity(original_image, in_range='image', out_range=(0, 1))
    original_image_rgb = gray2rgb(original_image_normalized)  # 元の画像をRGBに変換
    original_image_rgb[edges, :] = [1, 1, 0]  # エッジの位置を黄色に変更 (R, G, B)
    
    # Convert the image to uint8
    original_image_rgb_uint8 = (original_image_rgb * 255).astype(np.uint8)
    
    # Save the image with overlaid edges in JPEG format
    output_path = os.path.join(output_edge_dir, os.path.splitext(filename)[0] + '_edges.jpg')
    imsave(output_path, original_image_rgb_uint8)
